<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](mlcourse.ai) – دورة مفتوحة للتعلم الآلي 
### <center> المؤلف: Александр Кацалап (الملقب ODS Slack: Alexkats)
    
## <center> التنبؤ بأسعار العقارات في سوق الإسكان في ملبورن



سأقوم في هذا المشروع بتحليل بيانات **سوق الإسكان في ملبورن**، التي جمعها [توني بينو](https://www.kaggle.com/anthonypino) ونشرها على [Kaggle](https://www.kaggle.com/anthonypino/melbourne-housing-market/home).
تم استخلاص هذه البيانات من النتائج المتاحة للعامة والتي يتم نشرها كل أسبوع من المورد العقاري [Domain.com.au](https://www.domain.com.au). تتضمن مجموعة البيانات العنوان ونوع العقار والضاحية وطريقة البيع والغرف والسعر والوكيل العقاري وتاريخ البيع والمسافة من منطقة الأعمال المركزية. - وسط ملبورن.
الغرض من هذا المشروع هو بناء نموذج يسمح بالتنبؤ بسعر العقارات في سوق المنازل في ملبورن، بناءً على خصائصه. ** إذن، مهمتنا هي مهمة الانحدار. **
قد يكون من المفيد معرفة السعر الفعلي للعقار في الحالات التالية:
- أنت ***بائع عقار*** وتريد بيعه في أسرع وقت ممكن. لا تريد بيعه بسعر منخفض وتخسر ​​أموالك. ولا ترغب في العثور على مشتري خلال فترة طويلة بسبب ارتفاع سعر العقار.
- أنت ***مشتري عقار*** وترغب في شراء منزل جيد بسعر جيد ولا تريد أن تدفع مبالغ زائدة.
- أنت **وكالة عقارية** مثل [Domain.com.au](https://www.domain.com.au) وتريد إزالة الإعلانات التي تحتوي على كائنات مشبوهة على موقع الويب الخاص بك. على سبيل المثال، إذا كان سعر إعلان البيع منخفضًا جدًا مقارنة بأشياء لها نفس الخصائص تقريبًا، فقد يكون ذلك احتيالًا. سيساعد التنبؤ بالأسعار الفعلية على اكتشاف مثل هذه الإعلانات وإزالتها و**لن تخسر عملائك**.
	



### الجزء الأول. وصف مجموعة البيانات والميزات


تحتوي مجموعة البيانات على معلومات حول مبيعات العقارات في ملبورن خلال الفترة **من يناير 2016 إلى أكتوبر 2018.**
لنقم بتحميل مجموعة البيانات ووصف الميزات المحددة:


In [ ]:
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

In [ ]:
import numpy as np

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
warnings.simplefilter("ignore")

In [ ]:
full_data = pd.read_csv('Melbourne_housing_FULL.csv', parse_dates=['Date'])
full_data.head()

In [ ]:
full_data = full_data[full_data['Date'] <= '2018-04-01']


كل كائن في مجموعة البيانات (كل صف) هو عقار مباع بخصائصه الخاصة ومعلومات إضافية كبائع ونوع بيع وبيع.
دعنا نحصل على معلومات حول أنواع الأعمدة والقيم التي تم تخطيها في مجموعة البيانات:


In [ ]:
full_data.info()


كما نرى، تحتوي البيانات على 21 عمودًا، العديد منها يحتوي على قيم مفقودة.



### 1.1 وصف الميزات



دعونا نعطي وصفًا أكثر تفصيلاً حول معنى الأعمدة:



**الضاحية**: اسم ضاحية في ملبورن
**العنوان**: العنوان
**الغرف**: عدد الغرف
**غرفة النوم2**: عدد الغرف (من مصدر مختلف)
**السعر**: السعر بالدولار الأسترالي. إنها ***القيمة المستهدفة*** في مهمتنا
**الطريقة** - نوع طريقة البيع:
- S - الممتلكات المباعة؛ 
- SP - الممتلكات المباعة من قبل؛ 
- PI - الملكية التي تم تمريرها؛ 
- PN - تم بيعه مسبقًا ولم يتم الكشف عنه؛ 
- SN - تم بيعه ولم يتم الكشف عنه؛ 
- VB - عرض البائع؛ 
- W - تم سحبه قبل المزاد؛ 
- SA - بيعت بعد المزاد؛ 
- SS - تم بيعه بعد عدم الكشف عن سعر المزاد. 
**النوع** - نوع العقار:
- ح - منزل، كوخ، فيلا، شبه، تراس؛ 
- ش - وحدة مزدوجة؛ 
- ر - تاون هاوس؛ 
**SellerG**: وكيل عقارات
**التاريخ**: تاريخ البيع
** المسافة **: المسافة من C.B.D. (مركز ملبورن) بالكيلومترات
**اسم المنطقة**: المنطقة العامة (الغرب، الشمال الغربي، الشمال، الشمال الشرقي ...إلخ)
**عدد العقارات**: عدد العقارات الموجودة في الضاحية.
**الحمام**: عدد الحمامات
**السيارة**: عدد مواقف السيارات
**حجم الأرض**: حجم الأرض بالأمتار
**مساحة البناء**: حجم المبنى بالأمتار
**سنة البناء**: سنة بناء المنزل
**منطقة المجلس**: مجلس إدارة المنطقة
**خط العرض**: إحداثيات خط العرض للملكية**خط الطول**: إحداثيات خط الطول للملكية



من هذا الوصف يمكن تصنيف معظم هذه الميزات إلى فئوية ورقمية (متواصلة):
***الميزات الفئوية:***
- الضاحية، النوع، الطريقة، البائع G، الرمز البريدي، منطقة المجلس، اسم المنطقة
*** الميزات الرقمية: ***
- الغرف، التاريخ، المسافة، غرفة النوم 2، الحمام، السيارة، حجم الأرض، منطقة البناء، سنة البناء، خط العرض، خط الطول، عدد العقارات
***القيمة المستهدفة:***
- السعر
هناك أيضًا بعض الميزات المعقدة، التي ***لا يمكن تصنيفها بالتأكيد*** حسب نوعها ويجب تحويلها قبل استخدامها في النمذجة:
- العنوان، الرمز البريدي



### الجزء الثاني. تحليل البيانات الاستكشافية



كما نرى، هناك قيم مفقودة في هدفنا. والسبب هو أن **طريقة** بيع هذه العقارات كانت من بين **PN, SS**. ولم تكن هذه الأساليب تعني الإفصاح عن سعر البيع. لذلك يتعين علينا إزالة هذه الكائنات من مجموعة البيانات عندما نقوم ببناء نموذجنا. ولكن قبل ذلك نحفظها لحساب بعض الإحصائيات ولإسناد القيم المفقودة.
لنفصل مجموعة البيانات إلى جزأين: بالقيمة المستهدفة وبدون القيمة المستهدفة:



وتقسيم البيانات مع عدم فقدان الهدف إلى بيانات الميزات والبيانات المستهدفة:


In [ ]:
X_full = full_data.copy()
y_full = full_data['Price']


### 2.1 ميزات التفاعلات وتأثيرها على الهدف



دعونا نشرح تأثير الميزات على المتغير المستهدف - **السعر**.1. من الواضح أن الميزات ** الغرف، غرفة النوم 2، الحمام، السيارة، مساحة الأرض، مساحة البناء ** تتناسب بشكل مباشر مع سعر المنزل. لذلك، من المتوقع أن يكون لديهم ارتباط كبير بالهدف.
 
2. **تاريخ** البيع من المحتمل أن يكون له تأثير موسمي على السعر: على سبيل المثال، هناك موسم منخفض في الصيف وموسم مرتفع في الشتاء. 
 
3. قد يكون للمسافة** من اتفاقية التنوع البيولوجي تأثير معقد وغير خطي على السعر. فمن ناحية يجب أن يكون السعر هو الأعلى في المركز ويجب أن ينخفض ​​عند الانتقال إلى الضواحي. من ناحية أخرى، يوجد في وسط المدينة الكبيرة بيئة سيئة وصاخبة للغاية. 
  
4. يمكن إجراء تفكير مماثل بالنسبة إلى **سنة البناء** للبناء. فمن ناحية يجب أن يكون السعر هو الأعلى بالنسبة للمباني والمنازل الجديدة. ومن ناحية أخرى، قد تكون المباني القديمة جدًا آثارًا معمارية ولها قيمة تاريخية، لذلك قد تكون المباني القديمة جدًا ذات أسعار مرتفعة جدًا.
 
5. الميزات **الضاحية، الرمز البريدي، اسم المنطقة** تميز مواقع المنازل في المدينة، ونتيجة لذلك، حالة الجريمة وإمكانية الوصول إلى وسائل النقل. لذلك، يجب أن تؤثر هذه الميزات ومجموعاتها المختلفة على سعر المنزل.
6. **منطقة المجلس** قد تميز جودة العمل الحكومي المحلي. وتعتمد درجة الرفاهية على هذا العمل، ونتيجة لذلك، تعتمد أسعار المنازل في مناطق مختلفة.
7. **نوع** العقار مهم بالتأكيد، لأن امتلاك كوخ أو فيلا أكثر تكلفة من دوبلكس مع الجيران.



### 2.2 تحليل القيمة المستهدفة



احفظ القيمة المستهدفة **السعر** بدون NaNs في المتغير $y$ للتحليل:


In [ ]:
y = full_data[full_data.Price.notnull()]['Price']


لنرسم توزيع القيمة المستهدفة:


In [ ]:
plt.figure(figsize=(14, 7))
sns.distplot(y)
plt.grid()
plt.title('Price distribution');

إنه ليس توزيعًا طبيعيًا، لذا ليس من الجيد التنبؤ بهذه القيمة مباشرة. لذلك نحاول أخذ **لوغاريتم الهدف** ورسم توزيع هذه القيمة المحولة:
$$ \widehat{y} = ln (y + 1) $$


In [ ]:
plt.figure(figsize=(14, 7))
sns.distplot(np.log( 1.0 + y ))
plt.grid()
plt.title('Logarithm of Price distribution');


لذلك دعونا نجري اختبارات إحصائية للحياة الطبيعية والتواء التوزيع لـ $\widehat{y} $. 
استخدم **اختبارات الحياة الطبيعية لشابيرو-ويلك وكولموجوروف-سميرنوف**:


In [ ]:
from scipy.stats import shapiro, kstest, probplot, skew
test_stat, p_value = shapiro(np.log(y))
test_stat, p_value

In [ ]:
test_stat, p_value = kstest(np.log(y), cdf='norm')
test_stat, p_value


** مؤامرة QQ لـ $\widehat{y}$**. 
للتوزيع الطبيعي المثالي، تقع جميع النقاط الزرقاء على الخط الأحمر.


In [ ]:
plt.figure(figsize=(7,7))
probplot(np.log(y), dist='norm', plot=plt);
plt.grid()


**الانحراف** اختبار. للتوزيع المتماثل نتيجة اختبار التواء تساوي الصفر.


In [ ]:
skew(np.log(y))


ومع ذلك، فإن توزيع $\widehat{y}$ غير متماثل إلى حدٍ ما، وتم اجتياز QQ-plot وكلا الاختبارين الطبيعيين **يسمح لنا بالعمل مع القيمة $\widehat{y}$ كما هو الحال مع القيمة الموزعة العادية.**
لذلك، سنعمل مع القيمة المستهدفة $y$ على النحو التالي:
1. تدريب النماذج على هدف التحويل $y^* = ln(y+1)$ 
2. قم بالتنبؤ كـ $\widehat{y}_*$ 
3. قم بإجراء التحويل العكسي إلى الهدف الأصلي: $\widehat{y} = e^{\widehat{y}_*} - 1$ 
4. تحقق من جودتها عن طريق حساب بعض المقاييس $f(y, \widehat{y})$، والتي سيتم اختيارها لاحقًا.



### 2.3 معالجة القيم المفقودة



نظرًا لوجود الكثير من البيانات المفقودة، سنحاول أولاً ملء هذه البيانات.


In [ ]:
X_full.info()


املأ الحقول اسم المنطقة، عدد العقارات، منطقة المجلس، الرمز البريدي. 
دعونا نلقي نظرة على الكائن مع الرمز البريدي المفقود:


In [ ]:
X_full[X_full.Postcode.isnull()]


معظم الحقول مفقودة، لذا من الأفضل إسقاط هذا الكائن:


In [ ]:
X_full = X_full[~X_full.Postcode.isnull()]

In [ ]:
X_full.Postcode = X_full.Postcode.astype(int)


بالنسبة للكائنات ذات اسم المنطقة، وعدد الممتلكات، ومنطقة المجلس المفقودة، نحصل على نفس الحالة: معظم الحقول مفقودة، لذلك نقوم بإسقاطها أيضًا:


In [ ]:
X_full[X_full.Regionname.isnull()]

In [ ]:
X_full = X_full[~X_full.Regionname.isnull()]


تحقق من عدد القيم المفقودة في البيانات مرة أخرى:


In [ ]:
X_full.info()


تحقق من قيم YearBuilt. رسم بياني:


In [ ]:
plt.figure(figsize=(10,5))
X_full.YearBuilt.hist(bins=100)

plt.text(x=1200, y = 1200, s='Min built year = {}'.format(X_full.YearBuilt.min()))
plt.text(x=1200, y = 1100, s='Max built year = {}'.format(X_full.YearBuilt.max()))
plt.title('Year built');

هناك قيم خاطئة، بما في ذلك القيم من المستقبل :)
اضبطهم على NaN، ثم املأهم بالمفقودين الآخرين.


In [ ]:
X_full[(X_full.YearBuilt < 1800) | (X_full.YearBuilt > 2018)]

In [ ]:
X_full.loc[(X_full.YearBuilt < 1800) | (X_full.YearBuilt > 2018), 'YearBuilt'] = np.nan


لاحظ أن هناك الكثير من قيم خطوط الطول والعرض المفقودة. ولكن يمكننا استعادتها من خلال قيم العنوان (باستخدام الشارع) والرمز البريدي واسم المنطقة والضاحية ومنطقة المجلس. 
حدد الكائنات ذات خطوط الطول والعرض المملوءة، وابحث عن القيم المتوسطة لها عن طريق التجميع حسب هذه القيم:


In [ ]:
coords_features = ['Address', 'Postcode', 'Regionname', 'Suburb', 'CouncilArea', 'Lattitude', 'Longtitude']

In [ ]:
coords_data = X_full[~((X_full.Lattitude.isnull()) & (X_full.Longtitude.isnull()))][coords_features]

In [ ]:
coords_data.head()


استخراج اسم الشارع من العنوان:


In [ ]:
coords_data['Address_splitted'] = coords_data.Address.str.split(' ')
coords_data['Street'] = coords_data.Address_splitted.apply(lambda s: s[1])

In [ ]:
group_features = ['Regionname','Suburb','CouncilArea']

In [ ]:
coords_data_mean = coords_data\
                    .groupby(group_features)['Lattitude','Longtitude']\
                    .mean()\
                    .reset_index()\
                    .rename(columns={'Lattitude': 'Lat_new', 'Longtitude': 'Lon_new'})

In [ ]:
coords_data_mean.head()


أضف الآن اسم الشارع إلى مجموعة البيانات الخاصة بنا وادمجه مع **coords_data_mean** :


In [ ]:
X_full['Address_splitted'] = X_full.Address.str.split(' ')
X_full['Street'] = X_full.Address_splitted.apply(lambda s: s[1])
X_full['HouseNumber'] = X_full.Address_splitted.apply(lambda s: s[0])
X_full.drop('Address_splitted', axis=1, inplace=True)

In [ ]:
X_full_2 = pd.merge(X_full, coords_data_mean, on=group_features, how='left')

In [ ]:
X_full_2.head()


الآن استبدل Nans في **Lattitude** و **Longtitude** بقيمتين جديدتين **Lat_new** و **Lon_new**:


In [ ]:
X_full_2.loc[X_full_2.Lattitude.isnull(), 'Lattitude'] = X_full_2['Lat_new']
X_full_2.loc[X_full_2.Longtitude.isnull(), 'Longtitude'] = X_full_2['Lon_new']


وتحقق مما إذا كانت جميع NaNs في Lattitude وLongtitude ممتلئة:


In [ ]:
X_full_2[X_full_2.Lattitude.isnull()].shape[0]


لا، هناك 87 قيمة شاغرة، نظرًا لوجود بعض أسماء المنطقة والضاحية ومنطقة المجلس، عندما كانت قيم خط العرض وخط الطول مفقودة تمامًا. لذا، نظرًا لأن عدد هذه الكائنات صغير جدًا مقارنة بحجم مجموعة البيانات، فيمكننا ملؤها بالمتوسط:


In [ ]:
X_full_2.Lattitude = X_full_2.Lattitude.fillna(X_full_2.Lattitude.mean())
X_full_2.Longtitude = X_full_2.Longtitude.fillna(X_full_2.Longtitude.mean())

X_full_2.drop(['Lat_new','Lon_new'], axis=1, inplace=True)


التحقق من أعداد القيم المفقودة:


In [ ]:
X_full_2.info()


لذلك، لدينا ميزات مفقودة: ** غرفة النوم 2، الحمام، السيارة، حجم الأرض، مساحة البناء **



قبل ملء المفقود منها، دعونا نتحقق من توزيعاتها. قد تحتوي على زيوت يمكن أن يكون لها تأثير سيء على جودة التعبئة وعلى جودة النموذج في المستقبل.
دعونا نرسم توزيعات هذه الميزات:


In [ ]:
# Bathroom
X_full_2[~X_full_2.Bathroom.isnull()].Bathroom.hist(bins=11)
X_full_2[~X_full_2.Bathroom.isnull()].Bathroom.value_counts()


هناك كائنات بها 7 حمامات وأكثر! دعونا ننظر إليهم:


In [ ]:
X_full[X_full.Bathroom >= 8][['Suburb', 'Address', 'Rooms', 'Type', 'Method',  'Date',
       'Distance', 'Bedroom2', 'Bathroom', 'Car', 'Landsize', 'BuildingArea', 'YearBuilt', 'Price']]

In [ ]:
# Bedroom
X_full_2[~X_full_2.Bedroom2.isnull()].Bedroom2.hist(bins=15)
X_full_2[~X_full_2.Bedroom2.isnull()].Bedroom2.value_counts()


يبدو الأمر مريبًا للغاية، خاصة الأشياء، عندما تكون **كمية غرف النوم تساوي كمية الحمامات :)**
الشيء نفسه بالنسبة لعدد غرف النوم الصفرية.
لذا فالحل الأفضل **إسقاط الأشياء التي بها حمامات عددها أكثر من 6 وغرف نوم عددها أكثر من 8 (أو تساوي صفر):**


In [ ]:
X_full_3 = X_full_2[(X_full_2.Bathroom.isnull()) | (X_full_2.Bathroom <= 6)]
X_full_3 = X_full_3[(X_full_3.Bedroom2.isnull()) | ((X_full_3.Bedroom2 <= 8) & (X_full_3.Bedroom2 > 0))]

تحقق من ميزة **السيارة**:


In [ ]:
# Car
X_full_3[~X_full_3.Car.isnull()].Car.hist(bins=9)
X_full_3[~X_full_3.Car.isnull()].Car.value_counts()

In [ ]:
X_full[X_full.Car > 9 ][['Suburb', 'Address', 'Rooms', 'Type', 'Method',  'Date',
       'Distance', 'Bedroom2', 'Bathroom', 'Car', 'Landsize', 'BuildingArea', 'YearBuilt', 'Price']].head()


لذلك، نرى أن الكائنات التي تحتوي على عدد أكبر من 9 من نقاط السيارات لها قيم Landsize كبيرة جدًا وأسعار منخفضة بشكل مثير للريبة. لذا، هذه **الكائنات لا تبدو مثل معظم الكائنات الأخرى**، وسنقوم بإسقاطها أيضًا:


In [ ]:
X_full_3 = X_full_3[(X_full_3.Car.isnull()) | (X_full_3.Car < 9)]


تحقق من ميزة **حجم الأرض**:


In [ ]:
# Landsize
X_full_3[~X_full_3.Landsize.isnull()].Landsize.hist();


لذلك، التوزيع منحرف جدًا، بسبب القيم المتطرفة ذات القيم الضخمة جدًا. فلنبحث عن **القيمة المتوسطة** ونحسب **99.9% الكمية لميزة Landsize**. يمكنك أيضًا العثور على الكائنات ذات الأحجام العشرة الأولى في مجموعة البيانات الخاصة بنا:


In [ ]:
# Find mean value of Landsize:
X_full_3.Landsize.mean()

In [ ]:
# Find 99.9 % quantile of Landsize:
q_99 = X_full_3[~X_full_3.Landsize.isnull()].Landsize.quantile(0.999)
q_99

In [ ]:
# Find top-10 objects with biggest Landsize:
top_10_landsizes = sorted(X_full_3[~X_full_3.Landsize.isnull()].Landsize, reverse=True)[:10]

In [ ]:
top_10_landsizes

In [ ]:
X_full_3[X_full_3.Landsize.isin(top_10_landsizes)][['Suburb', 'Address', 'Rooms', 'Type', 'Method',  'Date',
       'Distance', 'Bedroom2', 'Bathroom', 'Car', 'Landsize', 'BuildingArea', 'YearBuilt', 'Price']]


لذلك، تقع جميعها تقريبًا بعيدًا عن منطقة الأعمال المركزية (يبلغ متوسط المسافة حوالي 11 كم). كما هو الحال مع القيم المتطرفة الأخرى، سنسقط الكائنات ذات حجم Landsize أكثر من 99.9%، لأنها سيكون لها تأثير سيء على جودة النموذج:


In [ ]:
X_full_3 = X_full_3[(X_full_3.Landsize.isnull()) | (X_full_3.Landsize < q_99)]


ارسم التوزيع بعد إزالة القيم المتطرفة:


In [ ]:
X_full_3[~X_full_3.Landsize.isnull()].Landsize.hist(bins=50);


تحقق من ميزة **BuildingArea**:


In [ ]:
# Landsize
X_full_3[~X_full_3.BuildingArea.isnull()].BuildingArea.hist(bins=50);


لدينا نفس الموقف كما هو الحال مع **Landsize**، لذلك دعونا نكرر الإجراء الخاص بإسقاط القيم المتطرفة باستخدام ميزة **BuildingArea**:


In [ ]:
# Find 99 % quantile of Landsize:
q_99 = X_full_3[~X_full_3.BuildingArea.isnull()].BuildingArea.quantile(0.99)
q_99

In [ ]:
X_full_3 = X_full_3[(X_full_3.BuildingArea.isnull()) | (X_full_3.BuildingArea < q_99)]


ارسم التوزيع بعد إزالة القيم المتطرفة:


In [ ]:
X_full_3[~X_full_3.BuildingArea.isnull()].BuildingArea.hist(bins=50);


لاحظ أن BuildingArea لا يمكن أن تساوي الصفر! (على عكس Landsize). ولكن لدينا عدة قيم صفرية في ميزة BuildingArea. يبدو أن هذا خطأ، لذا دعونا نغير هذه القيم الصفرية إلى NaNs:


In [ ]:
X_full_3.loc[X_full_3.BuildingArea == 0, 'BuildingArea'] = np.nan


لذلك، قمنا بإسقاط كائنات ذات قيم غير طبيعية للعديد من الميزات. 
**أعتقد أنه في مهمة العمل الحقيقية علينا فقط بناء نماذج أخرى وفصل النماذج لكل مجموعة من هذه الأشياء. ولكن هدفنا في هذه المهمة هو بناء نموذج واحد لمعظم الكائنات الموجودة في مجموعة البيانات لدينا.**


In [ ]:
X_full_3.info()


** سيكون من الأصح معالجة NaNs في الميزات كما فعلنا مع Lattitude وLongtitude من خلال حساب القيم المتوسطة في مجموعات ذات كائنات متشابهة بدون NaNs واستخدام تلك القيم لملء NaNs. **لكن دعونا نستخدم **SimpleImputor** من Sklearn لتوفير الوقت والتنوع :)


In [ ]:
import sklearn
from sklearn.impute import SimpleImputer

In [ ]:
imputer_mean = SimpleImputer(missing_values=np.nan, strategy='median')


حدد الميزات لإسنادها:


In [ ]:
features_with_nans = X_full_3.columns[X_full_3.isnull().any()].tolist()
features_with_nans.remove('Price')

In [ ]:
X_full_3.reset_index(drop=True, inplace=True)
X_to_impute = X_full_3[features_with_nans].copy()

In [ ]:
X_imputed_array = imputer_mean.fit_transform(X_to_impute)
X_imputed = pd.DataFrame(data=X_imputed_array, columns=features_with_nans)


إنشاء مجموعة بيانات جديدة بالقيم المحسوبة:


In [ ]:
X_full_4 = pd.concat([X_full_3.drop(features_with_nans, axis=1), X_imputed], axis=1)

In [ ]:
X_full_4.info()


لذا، الآن لا يوجد NaNs في قيم ميزاتنا. 
** ولكن هناك NaNs في القيمة المستهدفة - السعر **. 
قبل بناء النموذج، سنقوم بإسقاط الكائنات ذات الأهداف المفقودة.



الآن دعونا نقسم بياناتنا إلى ميزات dataframe وناقل الهدف:


In [ ]:
data_total = X_full_4.copy()


أضف **الشارع** و**رقم المنزل** مرة أخرى:


In [ ]:
data_total['Address_splitted'] = data_total.Address.str.split(' ')
data_total['Street'] = data_total.Address_splitted.apply(lambda s: s[1])
data_total['HouseNumber'] = data_total.Address_splitted.apply(lambda s: s[0])
data_total.drop(['Address_splitted', 'Address'], axis=1, inplace=True)

In [ ]:
data_total.YearBuilt = data_total.YearBuilt.astype(int)
data_total.Bedroom2 = data_total.Bedroom2.astype(int)
data_total.Bathroom = data_total.Bathroom.astype(int)
data_total.Car = data_total.Car.astype(int)

In [ ]:
data = data_total[~data_total.Price.isnull()]

In [ ]:
X_total = data_total.drop('Price', axis=1)

In [ ]:
# Features for objects with price only
X = data_total[~data_total.Price.isnull()].drop('Price', axis=1)

In [ ]:
# Target vector
y = data_total[~data_total.Price.isnull()]['Price']


التحقق من أشكال إطارات البيانات:


In [ ]:
X.shape, y.shape


## الجزء 3. التحليل البصري للميزات



دعونا نقسم ميزاتنا إلى **فئوية** و**عددية**:


In [ ]:
numerical_features = ['Rooms','Distance', 'Propertycount', 
                      'Bedroom2', 'Bathroom', 'Car', 'Landsize', 
                      'BuildingArea', 'YearBuilt', 'HouseNumber']

In [ ]:
cat_features = ['Suburb', 'Address','Type', 'Method', 'SellerG','CouncilArea','Regionname']


### 3.1 إعادة صياغة الميزات العددية



دعونا نستخدم **seaborn Pairplot** لتصور العلاقات بين السمات العددية:


In [ ]:
sns.pairplot(data=data[numerical_features + ['Price']]);


لذلك، دعونا نرسم أكثرها إثارة للاهتمام بشكل منفصل: 


In [ ]:
sns.pairplot(data=data[numerical_features + ['Price']], 
             y_vars=['Price'], 
             x_vars=['Rooms',  'Distance',
                     'Car', 'Landsize', 'BuildingArea', 'YearBuilt']);

In [ ]:
sns.pairplot(data=data[numerical_features + ['Price']], 
             y_vars=['Distance'], 
             x_vars=['Landsize', 'BuildingArea', 'YearBuilt']);


من هذه المؤامرات يمكننا أن نستنتج أن السعر يتناسب عكسيا مع المسافة من اتفاقية التنوع البيولوجي. المزيد من المباني القديمة أقرب إلى منطقة الأعمال المركزية. والمنازل ذات المساحة الأرضية الأكبر تكون بعيدة عن منطقة الأعمال المركزية.
لذا فإن هذه الاستنتاجات كانت متوقعة ومتوافقة مع الواقع.



لنرسم **مصفوفة الارتباط** للميزات العددية والهدف:


In [ ]:
plt.figure(figsize=(10,10))
sns.heatmap(data[numerical_features + ['Price']].corr(), annot=True)


لذلك، **الغرف، منطقة المبنى، غرفة النوم 2، الحمام، السيارة** ترتبط ارتباطًا إيجابيًا بـ **السعر** كما هو متوقع.



قد تحتوي ميزات مثل **YearBuilt** على تبعيات أكثر تعقيدًا مع السعر، لأنها تحتوي على قيم ميزات فئوية مختلفة (على سبيل المثال **Type** و**Regionname**). للتحقق من هذه الفرضية علينا أن نحلل هذه وغيرها من الميزات الموسيقية بمزيد من التفصيل. 



#### 3.2 إعادة تصميم الميزات الفئوية مع الهدف


أستخدم **seaborn boxplot** لتصور توزيعات الأسعار في مجموعات مختلفة من ميزات القطط



لننظر كيف يتم توزيع الأسعار حسب خاصية **النوع**:


In [ ]:
plt.figure(figsize=(12,8))
sns.boxplot(x='Type', y='Price',
            data=data);
plt.ylim((0, 0.5*1e7))
plt.grid()
plt.show()


نفس المؤامرة لميزة **اسم المنطقة**:


In [ ]:
plt.figure(figsize=(18,8))

sns.boxplot(x='Regionname', y='Price',
            data=data);
plt.ylim((0, 0.4*1e7))
plt.grid()
plt.show()


دعونا نجمع بين هاتين المخططين لتصور توزيعات الأسعار، مجمعة حسب **اسم المنطقة** و **النوع** في وقت واحد:


In [ ]:
plt.figure(figsize=(18,12))

sns.boxplot(x='Regionname', y='Price',
            hue='Type',
            data=data);
plt.ylim((0, 0.4*1e7))
plt.grid()
plt.show()


ومن المثير جدًا رسم نفس الشيء بالنسبة لـ **Distance**:


In [ ]:
plt.figure(figsize=(18,12))

sns.boxplot(x='Regionname', y='Distance',
            hue='Type',
            data=data);
plt.grid()
plt.show()


Boxplot لـ **السعر** مع التجميع حسب **منطقة المجلس **:


In [ ]:
plt.figure(figsize=(18,12))
sns.boxplot(y='CouncilArea', x='Price', data=data);
plt.xlim((0, 0.4*1e7))
plt.grid()
plt.show()


دعونا نتصور توزيعات الأسعار، مجمعة حسب **الطريقة** و **النوع** في وقت واحد. يسمح بفهم تأثير **طريقة** الشراء على سعر المنزل في كل **النوع**:


In [ ]:
plt.figure(figsize=(18,12))

sns.boxplot(x='Type', y='Price',
            hue='Method',
            data=data);
plt.ylim((0, 0.4*1e7))
plt.grid()
plt.show()


لمعرفة عدد العناصر التي تحتوي على مناطق مختلفة، وعدد العناصر التي تم بيعها بطرق مختلفة، فلنرسم **countplots**:


In [ ]:
plt.figure(figsize=(18,12))
sns.countplot(data=data, hue='Type', y='Regionname');
plt.grid()

plt.figure(figsize=(18,12))
sns.countplot(data=data, hue='Method', y='Regionname');
plt.grid()


وبالعودة إلى الميزات العددية، فلنرسم **الأسعار** التوزيعات مجمعة حسب **الغرف، والسيارة، وسنة البناء، وغرفة النوم 2، والحمام ** والقط. ميزة ** النوع ** :


In [ ]:
for f in ['Rooms', 'Car', 'Bedroom2', 'Bathroom']:
    plt.figure(figsize=(18,6))
    sns.boxplot(y='Price', x=f,
                hue='Type',
                data=data);
    plt.ylim((0, 0.4*1e7))
    plt.grid()
    plt.show()


مؤامرة **YearBuilt** التوزيعات مجمعة حسب **اسم المنطقة** و **النوع**:


In [ ]:
plt.figure(figsize=(18,10))
sns.boxplot(y='YearBuilt', x='Regionname',
            hue='Type',
            data=data);
plt.grid()
plt.show()


توزيعات قطعة الأرض **السعر** مجمعة حسب **سنة البناء** :


In [ ]:
plt.figure(figsize=(18,10))
sns.boxplot(y='Price', x='YearBuilt', data=data);
plt.xticks(rotation=90)
plt.ylim((0, 0.6*1e7))

plt.show()


متوسط **السعر**، مجمعًا حسب **سنة البناء**


In [ ]:
prices_by_yearbuilt = data[['YearBuilt', 'Price']]\
.groupby('YearBuilt')\
.agg(['mean'])

In [ ]:
plt.figure(figsize=(12,8))
plt.scatter(x=prices_by_yearbuilt.index,y=prices_by_yearbuilt.values[:,0])
plt.xticks(rotation=90)

plt.grid()
plt.show()


توجد *إحداثيات جغرافية* في مجموعة البيانات لدينا: ميزات **Longtitude** و **Lattitude** . حتى نتمكن حرفيًا من رسم خريطة لبعض الميزات الفئوية! فلنفعل ذلك من أجل **'Suburb'، و'Postcode'، و'CouncilArea'، و'Regionname'** وحاول **'Distance'**:


In [ ]:
for feature in ['Suburb', 'Postcode','CouncilArea', 'Regionname', 'Distance']:

    plt.figure(figsize=(15,10))
    if feature == 'Postcode':
        feature_unique_values = data[feature].unique()
    else:
        feature_unique_values = sorted(data[feature].unique())
    colors = sns.color_palette("hls", len(feature_unique_values))
    for i, cat_value in enumerate(feature_unique_values):
        plt.scatter(x=data[data[feature] == cat_value]['Longtitude'],
                    y=data[data[feature] == cat_value]['Lattitude'], c=colors[i]);
    
    plt.title(feature)
    if feature in ['CouncilArea', 'Regionname']:
        plt.legend(feature_unique_values);
    plt.grid()
    plt.show()


#### 3.3 ميزة التاريخ وإعادته إلى الهدف وميزات أخرى



هناك تاريخ البيع في مجموعة البيانات. لنستخرج منه السنة والشهر وأسعار الأراضي وعدد المبيعات مجمعة حسب السنة والشهر:


In [ ]:
data['Month'] = data.Date.dt.month
data['Year'] = data.Date.dt.year

In [ ]:
plt.figure(figsize=(18,12))

sns.countplot(x='Month',
              hue='Year',
              data=data);
plt.grid()
plt.show()

### 3.4 الاستنتاجات



دعنا نستأنف افتراضاتنا من **ص.2** ونضيف استنتاجات بشأنها بعد تحليل البيانات المرئية:1. من الواضح أن الميزات ** الغرف، غرفة النوم 2، الحمام، السيارة، مساحة الأرض، مساحة البناء ** تتناسب بشكل مباشر مع سعر المنزل. لذلك، من المتوقع أن يكون لديهم ارتباط كبير بالهدف.
<font color = 'green'> نعم، بشكل عام، بدون التجميع حسب القطط. المستقبل هذا صحيح. </font>
 
2. **تاريخ** البيع من المحتمل أن يكون له تأثير موسمي على السعر: على سبيل المثال، هناك موسم منخفض في الصيف وموسم مرتفع في الشتاء. 
<font color = 'red'> لا، لا يوجد موسمية خلال العام. هناك بالأحرى **اتجاه صعودي** خلال كل الفترة في مجموعة البيانات. </font>
 
3. قد يكون للمسافة** من اتفاقية التنوع البيولوجي تأثير معقد وغير خطي على السعر. فمن ناحية يجب أن يكون السعر هو الأعلى في المركز ويجب أن ينخفض ​​عند الانتقال إلى الضواحي. من ناحية أخرى، يوجد في وسط المدينة الكبيرة بيئة سيئة وصاخبة للغاية. 
<font color = 'red'> لا، هناك في الغالب تبعية خطية بين **المسافة** و**السعر**. تم تأكيد ذلك من خلال مخطط الزوج ومعامل الارتباط السلبي </font>
  
4. يمكن إجراء تفكير مماثل بالنسبة إلى **سنة البناء** للبناء. فمن ناحية يجب أن يكون السعر هو الأعلى بالنسبة للمباني والمنازل الجديدة. ومن ناحية أخرى، قد تكون المباني القديمة جدًا آثارًا معمارية ولها قيمة تاريخية، لذلك قد تكون المباني القديمة جدًا ذات أسعار مرتفعة جدًا.
<font color = 'green'> نعم، هناك حقًا تبعية معقدة غير خطية: لا يمكننا رؤية تبعية خطية على مخطط الزوج، ولكن هناك معامل ارتباط سلبي صغير بشكل عام.   </font>
 
5. الميزات **الضاحية، الرمز البريدي، اسم المنطقة** تميز مواقع المنازل في المدينة، ونتيجة لذلك، حالة الجريمة وإمكانية الوصول إلى وسائل النقل. لذلك، يجب أن تؤثر هذه الميزات ومجموعاتها المختلفة على سعر المنزل.
<font color = 'green'> نعم، يوجدأسعار مختلفة حقا في مناطق مختلفة. ولكنها بالأحرى نتيجة لقيمة **المسافة**. 
 </font>
6. **منطقة المجلس** قد تميز جودة العمل الحكومي المحلي. وتعتمد درجة الرفاهية على هذا العمل، ونتيجة لذلك، تعتمد أسعار المنازل في مناطق مختلفة.
<font color = 'green'> نعم، هناك بالفعل أسعار مختلفة في **منطقة المجلس** المختلفة. ولكن تمامًا مثل **الضاحية، الرمز البريدي، اسم المنطقة**، فهي نتيجة لقيمة **المسافة**. 
 </font>
7. **نوع** العقار مهم بالتأكيد، لأن امتلاك كوخ أو فيلا أكثر تكلفة من دوبلكس مع الجيران.
<font color = 'green'> صحيح تمامًا، هناك أسعار مختلفة حقًا.
 </font>



بالإضافة إلى ذلك، هناك أيضًا فرق كبير في السعر في طريقة البيع - ميزة **الطريقة**.



### الجزء الرابع. الأنماط والرؤى وخصائص البيانات 



دعونا نلخص الاستنتاجات حول البيانات، بناءً على الأجزاء السابقة.



1. المنازل بأنواعها المختلفة لها فروق كبيرة في الأسعار. لذلك، سنأخذ في الاعتبار ميزة **النوع** في نموذج التنبؤ الخاص بنا.
2. الميزات التي تتوافق مع المعايير المادية للمنازل - **'الغرف'، 'غرفة النوم 2'، 'الحمام'، 'السيارة'، 'حجم الأرض'، 'منطقة البناء'** - لها تأثير مباشر على السعر وفقًا لمبدأ "كلما زاد الحجم/العدد، زادت التكلفة". لذلك، هذه الميزات مهمة.
3. يوجد في الغالب تبعية خطية بين **المسافة** و**السعر**. لذلك، **المسافة** هي ميزة مهمة.4. الميزات ** الضاحية، الرمز البريدي، اسم المنطقة، منطقة المجلس ** تميز مواقع المنازل. ولكن، كما كان واضحًا على "خرائطنا"، فإن **الضاحية** و **الرمز البريدي** يقدمان نفس أقسام المدينة تقريبًا، لذلك لا نحتاج إلى كليهما. علاوة على ذلك، فإنها تقدم الكثير من التفصيل وقد تكون هذه المعلومات غير مفيدة في نموذجنا على عكس الميزات **اسم المنطقة، منطقة المجلس**، لذلك سنحاول أولاً **اسم المنطقة، منطقة المجلس** في نموذجنا.
5. كان الأمر غير متوقع، ولكن طريقة البيع - ميزة **الطريقة** - مهمة حقًا، حيث يمكننا ملاحظة هذا التأثير على جميع أنواع المنازل. لذلك، ستكون هذه معلومات مهمة في نموذجنا أيضًا.
6. الدمج مع **YearBuilt** ليس بالأمر السهل. بالنسبة للمنازل التي بنيت قبل عام 1950، هناك تباين كبير في الأسعار. من عام 1950 إلى عام 2013، كان متوسط ​​السعر أكثر استقرارًا. ولكن المنازل الجديدة، التي بنيت بعد عام 2013، لديها متوسط ​​سعر أعلى. لذا، هذه الميزة مهمة، ولكننا بحاجة إلى تحويلها قبل إضافتها إلى نموذجنا.



### الجزء الخامس. اختيار المقاييس



التنبؤ بالسعر هو مهمة **انحدار**. في بياناتنا **السعر** التوزيع له معامل انحراف كبير. وهذا يعني أن هناك قيمًا متطرفة - وهي نسبة صغيرة جدًا وبأسعار باهظة مقارنة بمعظم الأشياء الأخرى. هدفنا هو بناء نموذج دقيق لمعظم الأشياء بالأسعار المعتادة. كما ذكر أعلاه، بالنسبة للمنازل باهظة الثمن للغاية، يتعين علينا بناء نموذج منفصل. 
لذلك، في المهمة الحالية، سيتم إعطاء الأولوية في دقة التنبؤ للأغلبية الرئيسية من الكائنات.المقياس الجيد هنا هو **MAE (متوسط ​​الخطأ المطلق)**. بالمقارنة مع **(R)MSE (جذر متوسط ​​الخطأ التربيعي)**، **MAE** أقل عرضة للأخطاء الكبيرة في المنازل ذات الأسعار الكبيرة جدًا (غرامات أقل للأخطاء الأكبر من حيث الحجم المطلق، حيث يتم أخذ معامل الخطأ فقط، وليس مربعه)، والذي يتوافق مع شروط مهمتنا - الحصول على مقاييس النموذج الأكثر ملاءمة لمعظم الكائنات. لن تؤدي الأخطاء الكبيرة في المنازل ذات الأسعار الباهظة إلى تشويه **MAE** على عكس **(R)MSE**.



$$ MAE = \frac{1}{n} \sum_{i=1}^n \mid{y_i - \widehat{y_i}}\mid $$



من وجهة نظر التفسير، من الواضح أن **MAE** هو الفائز. **(R)MSE** لا تصف متوسط ​​الخطأ وحده ولها آثار أخرى يصعب اكتشافها وفهمها. في مهمتنا **MAE** لدينا تفسير *"متوسط ​​الخطأ بالدولار الأسترالي"*. من السهل جدًا شرح هذا الخطأ لكل مشتري أو بائع.



### الجزء السادس. اختيار النموذج



معظم الميزات الموجودة في بياناتنا، كما هو موضح سابقًا، لها تبعية خطية مع **السعر**. لذلك، هذا سبب وجيه لاستخدام ***نموذج الانحدار الخطي*** في مهمتنا. على الرغم من بساطتها، تتمتع النماذج الخطية بالعديد من المزايا في مهمتنا:
1. تركيب سريع جدًا 
2. فعال مع عدد كبير من الميزات (لدينا ميزات فئوية ذات قيم متعددة، لذلك بعد تشفير ساخن واحد على سبيل المثال، سنحصل على المئات منها)
3. التفسير الجيد: أهمية الميزة هي مجرد قيمة مطلقة لمعاملها في النموذج الخطي المجهز.



سنستخدم **انحدار Lasso** من وحدة sklearn. تتمتع Lasso بخاصية لطيفة لاختيار الميزات. للمقارنة فقط، سنحاول **الغابات العشوائية**.



### الجزء السابع. المعالجة المسبقة للبيانات


تمت معالجة القيم المتطرفة والمفقودة من قبل في الجزء الثاني. لذا في هذا الجزء، قمنا فقط بتقسيم بياناتنا إلى مجموعات تدريب ومجموعات تحقق ونجري تشفيرًا سريعًا واحدًا.



#### 7.1 تقسيم البيانات إلى أجزاء تدريب وتحكم



لدينا تبعية للوقت في بياناتنا، لذلك يتعين علينا إنشاء مجموعات بيانات التدريب والتحكم، بحيث يكون الحد الأقصى لتاريخ القطار في بيانات القطار أقل أو يساوي الحد الأدنى من البيانات في مجموعة بيانات التحكم. لذلك نقوم بفرز البيانات حسب التاريخ وتقسيمها إلى تدريب وتحكم بنسبة 7/3:


In [ ]:
data_sorted = data.sort_values(by='Date')
data_sorted.reset_index(inplace=True, drop=True)

In [ ]:
X = data_sorted.drop('Price', axis=1)

In [ ]:
y = data_sorted['Price']

y.reset_index(inplace=True, drop=True)

In [ ]:
split_index = int(0.7*X.shape[0])

X_train = data_sorted.loc[: split_index, :].drop('Price',axis=1)
X_valid = data_sorted.loc[split_index:, :].drop('Price',axis=1)

In [ ]:
y_train = y.loc[: split_index]
y_valid = y.loc[split_index:]

In [ ]:
# Check sizes of datasets
X_train.shape, X_valid.shape, y_train.shape, y_valid.shape


#### 7.2 قم بعمل ترميز ساخن واحد



بدلاً من OneHotEncoder الخاص بـ sklearn، دعنا نستخدم الوظيفة الخاصة، فهي أكثر ملاءمة، لأن قيم السلسلة موجودة في الميزات الفئوية (يعمل OneHotEncoder فقط مع الميزات الفئوية الصحيحة).
لنبدأ أولاً برمز cat.features فقط مع القيم الفئوية الصغيرة وإسقاط القيم الأخرى. لذلك، فليكن خط الأساس لدينا.


In [ ]:
def make_one_hot_encoding(X, features):
    X_ohe = pd.get_dummies(data=X, columns=features)
    return X_ohe

In [ ]:
#  Cat.features only with small categorical values
ohe_features = ['Type', 'Method', 'CouncilArea', 'Regionname']

X_ohe_train = make_one_hot_encoding(X_train, features=ohe_features)
X_ohe_valid = make_one_hot_encoding(X_valid, features=ohe_features)

In [ ]:
X_ohe_train.drop(['Suburb', 'SellerG', 'Date','Postcode','Lattitude', 
            'Longtitude', 'Propertycount', 'Street', 'HouseNumber'], axis=1, inplace=True)

X_ohe_valid.drop(['Suburb', 'SellerG', 'Date','Postcode','Lattitude', 
            'Longtitude', 'Propertycount', 'Street', 'HouseNumber'], axis=1, inplace=True)

In [ ]:
X_ohe_train.drop('YearBuilt', axis=1, inplace=True)
X_ohe_valid.drop('YearBuilt', axis=1, inplace=True)

In [ ]:
X_ohe_train.shape, X_ohe_valid.shape


#### 7.3 توحيد المعايير، خط الأنابيب



عندما نستخدم النموذج الخطي، **من الضروري توحيد بياناتنا. دعونا نستخدم **StandardScaler**:


In [ ]:
from sklearn.preprocessing import  StandardScaler

scaler = StandardScaler()


من المريح استخدام المقياس والنموذج في **خط الأنابيب**:


In [ ]:
from sklearn.pipeline import Pipeline


دعونا أيضًا نحدد وظائف لتحويل هدفنا إلى اللوغاريتم والعكس:


In [ ]:
def to_log(y):
    return np.log(1 + y)

In [ ]:
def from_log(y):
    return np.exp(y) - 1


### الجزء 8. التحقق من صحة وتعديل المعلمات الفائقة للنموذج



عند استخدام Lasso، يتم حل مهمة التحسين التالية:
$$ \sum_{i=1}^l \sum_{j=1}^n (w_j x_{ij} - y_*)^2 + 
\lambda \sum_{j=1}^n \mid{w_j}\mid     \longrightarrow min{{\substack{w}}}
$$
حيث $\lambda$ هو **المعلمة الفائقة للتنظيم**. عندما يكون $\lambda$ صغيرًا، يمكن أن يكون ناقل الأوزان كبيرًا **$l1$-norm**، أي معاملات عالية في قيم الميزات لدينا $x_{ij}$، ونتيجة لذلك، سيكون النموذج غير مستقر للغاية (سيكون له **تباين عالٍ**). ومع نمو $\lambda$، سوف تنخفض الأوزان واحدًا تلو الآخر، وسيكون النموذج أكثر استقرارًا، ولكن سيكون به **تحيز** عالي.لذا، فإن مهمتنا هي العثور على $\lambda$ الأمثل، بحيث يوفر أفضل جودة أثناء التحقق المتبادل.



في **RandomForest** سنقوم بضبط المعلمة الفائقة **max_deep**. عندما لا يتم تقييدها، يمكن أن تنمو الأشجار في الغابة بشكل عميق ومعقد للغاية، مما قد يؤدي إلى فرط التجهيز. 
*أعراض التجاوز هي خطأ صغير في مجموعة بيانات القطار وخطأ كبير في مجموعة بيانات التحقق من الصحة.*



لنقم باستيراد الوحدات ذات النموذج الخطي **Lasso** (نريد تحديد الميزات) و**RandomForest** - نموذج للتحكم والوظيفة لحساب MAE:


In [ ]:
from sklearn.metrics import mean_absolute_error as mae

from sklearn.linear_model import Lasso
from sklearn.ensemble import  RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.pipeline import Pipeline


قم بإنشاء كائن نموذجي، وقم بإنشاء **خط أنابيب** لجعل نموذج النمل المقياس مناسبًا في نفس الوقت.
لاحظ أننا نستخدم **TimeSeriesSplit** التحقق المتبادل! 
سيسمح هذا بمراعاة الاعتماد على الوقت في بياناتنا.


In [ ]:
lasso = Lasso(random_state=42)

In [ ]:
params_grid = {'lasso__alpha': np.logspace(-4, 4, 10)}

In [ ]:
pipe = Pipeline(steps=[('scaler', scaler), ('lasso', lasso)])

In [ ]:
model_grid = GridSearchCV(pipe, 
                          params_grid, 
                          cv=TimeSeriesSplit(max_train_size=None, n_splits=5))

In [ ]:
%%time
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    model_grid.fit(X_ohe_train, to_log(y_train))


دعونا نتصور معاملات ميزاتنا (أنا أستخدم الكود الموجود في مقالة الدرس 4):


In [ ]:
def visualize_coefficients(classifier, feature_names, n_top_features=25):
    # get coefficients with large absolute values 
    coef = classifier.coef_.ravel()
    positive_coefficients = np.argsort(coef)[-n_top_features:]
    negative_coefficients = np.argsort(coef)[:n_top_features]
    interesting_coefficients = np.hstack([negative_coefficients, positive_coefficients])
    # plot them
    plt.figure(figsize=(18, 8))
    colors = ["red" if c < 0 else "blue" for c in coef[interesting_coefficients]]
    plt.bar(np.arange(2 * n_top_features)+1, coef[interesting_coefficients], color=colors)
    feature_names = np.array(feature_names)
    
    plt.xticks(np.arange(1, 1 + 2 * n_top_features), 
               feature_names[interesting_coefficients], rotation=60, ha="right");
    plt.xlabel("Feature name")
    plt.ylabel("Feature weight")
    plt.title("LASSO feature importances")
    plt.grid()

In [ ]:
visualize_coefficients(model_grid.best_estimator_.steps[1][1], 
                       X_ohe_train.columns, 
                       n_top_features=20)


عمل تنبؤات بشأن بيانات القطار والتحقق من الصحة:


In [ ]:
y_pred_train = from_log(model_grid.predict(X_ohe_train))
y_pred_valid = from_log(model_grid.predict(X_ohe_valid))


الآن لنقم بإنشاء مؤامرة، عندما يكون لكل كائن إحداثي x $y$ (القيمة الحقيقية) وإحداثي y $\widehat{y}$ (القيمة المتوقعة)، احسب أيضًا MAE لمجموعات بيانات التدريب والاختبار و$ \lambda$. من الواضح أنه بالنسبة للنقاط النموذجية الجيدة يجب أن تحدد موقع الخط القطري تقريبًا:
(*سنقوم فيما يلي بإنشاء قطعة الأرض هذه فقط للأشياء التي تتراوح أسعارها بين 99%*)


In [ ]:
plt.figure(figsize=(12,12))
plt.xlim((0, y.quantile(0.99)))
plt.ylim((0, y.quantile(0.99)))
plt.scatter(y=y_pred_train,  x=y_train, c='blue', s=7)
plt.scatter(y=y_pred_valid,  x=y_valid, c='green', s=7)
plt.plot([0,3e6],[0, 3e6], 'r-')
plt.legend(['"Ideal model" diagonal','Predicted train data','Predicted validation data'])
plt.text(s=" MAE_train = {0:.2f}".format(mae(y_pred_train, y_train)), x=1e5, y=3e6)
plt.text(s=" MAE_valid = {0:.2f}".format(mae(y_pred_valid, y_valid)),  x=1e5, y=2.9e6)
plt.text(s="""$ \\lambda = {0:.4f} $""".format(model_grid.best_estimator_.steps[1][1].alpha), x=1e5, y=2.8e6)
plt.title("True VS Predicted values (LASSO)")
plt.grid()


لذلك، حصلنا على $MAE_{train} = 219484$ و$MAE_{valid} = 216917$.
دعونا نحاول أن نفعل ما هو أفضل من خلال إنشاء بعض الميزات الجديدة.



### الجزء 9. إنشاء ميزات جديدة ووصف لهذه العملية



لنقم بإنشاء مجموعات بيانات منفصلة لإضافة ميزات إليها:


In [ ]:
split_index = int(0.7*data.shape[0])

data_sorted = data.sort_values(by='Date')
data_sorted.reset_index(inplace=True, drop=True)

In [ ]:
X_train_2 = data_sorted.loc[: split_index, :]
X_valid_2 = data_sorted.loc[split_index:, :]

In [ ]:
y_train_2 = X_train_2['Price']
y_valid_2 = X_valid_2['Price']

In [ ]:
# Check shapes
X_train_2.shape, X_valid_2 .shape


#### 9.1 الميزات من YearBuilt


دعونا نقسم **YearBuilt** إلى 3 نطاقات وفقًا للاختلافات في متوسط ​​**السعر**. يمكننا رؤيته من مخطط التوزيعات من **الجزء 3.2**. دعونا نضيف أيضًا علمًا لعام 1970، نظرًا لوجود عدد كبير من المنازل التي تم بناؤها في ذلك العام:


In [ ]:
# Year Built

X_train_2['old'], X_train_2['middle'], X_train_2['new'] = 0, 0, 0
X_valid_2['old'], X_valid_2['middle'], X_valid_2['new'] = 0, 0, 0

X_train_2.loc[X_train_2['YearBuilt'] < 1942, 'old'] = 1
X_train_2.loc[(X_train_2['YearBuilt'] >= 1942) & (X_train_2['YearBuilt'] <= 2012), 'middle'] = 1
X_train_2.loc[X_train_2['YearBuilt'] > 2012, 'new'] = 1

X_valid_2.loc[X_valid_2['YearBuilt'] < 1942, 'old'] = 1
X_valid_2.loc[(X_valid_2['YearBuilt'] >= 1942) & (X_valid_2['YearBuilt'] <= 2012), 'middle'] = 1
X_valid_2.loc[X_valid_2['YearBuilt'] > 2012, 'new'] = 1

X_train_2['1970'], X_valid_2['1970'] = 0, 0
X_train_2.loc[X_train_2['YearBuilt'] == 1970, '1970'] = 1
X_valid_2.loc[X_valid_2['YearBuilt'] == 1970, '1970'] = 1


أيضًا، دعونا نحسب أعلى سنوات البناء الأكثر والأقل تكلفة (مرتبة حسب متوسط **السعر**). بالنسبة للموثوقية، سنأخذ في الاعتبار السنوات فقط التي تحتوي على 10 كائنات على الأقل: 


In [ ]:
count_by_year_built = X_train_2\
                    .groupby('YearBuilt')['Suburb']\
                    .count()\
                    .reset_index()\
                    .rename(columns={'Suburb':'HousesBuiltInYear'})

In [ ]:
mean_price_by_year_built = X_train_2.groupby('YearBuilt')['Price'].mean().reset_index()

In [ ]:
mean_price_by_year_built = pd.merge(mean_price_by_year_built, 
                                    count_by_year_built, on=['YearBuilt'])

In [ ]:
# The most expensive and cheapest houses by YearBuilt

top_year_built_by_price = mean_price_by_year_built\
                            .query("HousesBuiltInYear > 10")\
                            .sort_values(by=['Price'], ascending=False)\
                            .head(10)['YearBuilt']\
                            .tolist()

last_year_built_by_price = mean_price_by_year_built\
                            .query("HousesBuiltInYear > 10")\
                            .sort_values(by=['Price'], ascending=False)\
                            .tail(10)['YearBuilt']\
                            .tolist()

X_train_2['TopYearBuilt'], X_train_2['LastYearBuilt'] = 0, 0
X_valid_2['TopYearBuilt'], X_valid_2['LastYearBuilt'] = 0, 0

X_train_2.loc[X_train_2['YearBuilt'].isin(top_year_built_by_price), 'TopYearBuilt'] = 1
X_valid_2.loc[X_valid_2['YearBuilt'].isin(top_year_built_by_price), 'TopYearBuilt'] = 1

X_train_2.loc[X_train_2['YearBuilt'].isin(last_year_built_by_price), 'LastYearBuilt'] = 1
X_valid_2.loc[X_valid_2['YearBuilt'].isin(last_year_built_by_price), 'LastYearBuilt'] = 1


#### 9.2 ميزات من الشوارع



دعونا نكرر نفس الإجراء كما هو مذكور أعلاه لـ **الشارع**:


In [ ]:
# The most expensive and cheapest houses by Streets

price_by_street = X_train_2.groupby('Street')['Price'].mean().reset_index()
count_by_street = X_train_2.groupby('Street')['Suburb'].count().reset_index().rename(columns={'Suburb':'HousesCount'})

price_by_street = pd.merge(price_by_street, count_by_street, on=['Street'])

top_streets = price_by_street\
                .query("HousesCount > 10")\
                .sort_values(by=['Price'], ascending=False)\
                .head(50)['Street']\
                .tolist()

last_streets = price_by_street\
                .query("HousesCount > 10")\
                .sort_values(by=['Price'], ascending=False)\
                .tail(50)['Street']\
                .tolist()

X_train_2['TopStreet'], X_train_2['LastStreet'] = 0, 0
X_valid_2['TopStreet'], X_valid_2['LastStreet'] = 0, 0

X_train_2.loc[X_train_2['Street'].isin(top_streets), 'TopStreet'] = 1
X_valid_2.loc[X_valid_2['Street'].isin(top_streets), 'TopStreet'] = 1

X_train_2.loc[X_train_2['Street'].isin(last_streets), 'LastStreet'] = 1
X_valid_2.loc[X_valid_2['Street'].isin(last_streets), 'LastStreet'] = 1


#### 9.3 ترميز واحد ساخن للقطط الأخرى. الميزات وإسقاط الأعمدة الأخرى


In [ ]:
#  Cat.features
ohe_features_2 = ['Type', 'Method', 'CouncilArea', 'Regionname']

X_ohe_train_2 = make_one_hot_encoding(X_train_2, features=ohe_features_2)
X_ohe_valid_2 = make_one_hot_encoding(X_valid_2, features=ohe_features_2)

In [ ]:
X_ohe_train_2.shape, X_ohe_valid_2.shape

In [ ]:
cols_to_drop = ['Suburb','Price', 'SellerG', 'Date', 'Postcode','Lattitude', 
                'Longtitude', 'Propertycount', 'Street', 'HouseNumber', 'YearBuilt', 
                'Month', 'Year']

X_ohe_train_2.drop(cols_to_drop, axis=1, inplace=True)
X_ohe_valid_2.drop(cols_to_drop, axis=1, inplace=True)

In [ ]:
X_ohe_train_2.shape, X_ohe_valid_2.shape


دعونا نجهز نسخة من مجموعات البيانات بميزات جديدة لـ RandomForest:


In [ ]:
# Save copy for test RandomForestRegressor
X_ohe_train_rf = X_ohe_train_2.copy()
X_ohe_valid_rf = X_ohe_valid_2.copy()


#### 9.4 ميزات متعددة الحدود



في حين أن الانحدار الخطي هو مجرد مجموعة خطية من الميزات، فإن **الانحدار متعدد الحدود** مشابه جدًا، ولكنه يسمح بتركيبة خطية من قيم الميزات المرفوعة بدرجات متفاوتة. تتيح لنا هذه الحقيقة إنشاء نماذج أكثر تعقيدًا وغير خطية باستخدام نفس النماذج الخطية. تسمح هذه الخدعة باستعادة التبعيات الأكثر تعقيدًا بين الميزات والقيمة المستهدفة.
دعونا نحاول إضافة ميزات متعددة الحدود من الدرجة الثانية لبعض ميزات المصدر:


In [ ]:
from sklearn.preprocessing import PolynomialFeatures


features_to_poly = ['Regionname', 'Type', 'Street', 'Method', 'YearBuilt']
cols_to_poly = []
for col_name in X_ohe_train_2.columns:
    for f in features_to_poly:
        if f in col_name:
            cols_to_poly.append(col_name)

poly_generator = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)

X_train_poly_2 = poly_generator.fit_transform(X_ohe_train_2[cols_to_poly])
X_valid_poly_2 = poly_generator.transform(X_ohe_valid_2[cols_to_poly])

X_ohe_train_2 = np.hstack([X_ohe_train_2.drop(cols_to_poly, axis=1), X_train_poly_2])
X_ohe_valid_2 = np.hstack([X_ohe_valid_2.drop(cols_to_poly, axis=1), X_valid_poly_2])

In [ ]:
X_ohe_train_2.shape, X_ohe_valid_2.shape

In [ ]:
%%time
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
model_grid.fit(X_ohe_train_2, to_log(y_train_2))


قم مرة أخرى بعمل تنبؤات على القطار ومجموعات البيانات الصالحة ورسم الأسعار الحقيقية والمتوقعة:


In [ ]:
y_pred_train_2 = from_log(model_grid.predict(X_ohe_train_2))
y_pred_valid_2 = from_log(model_grid.predict(X_ohe_valid_2))

In [ ]:
plt.figure(figsize=(12,12))
plt.xlim((0, y.quantile(0.99)))
plt.ylim((0, y.quantile(0.99)))
plt.scatter(y=y_pred_train_2,  x=y_train, c='blue', s=7)
plt.scatter(y=y_pred_valid_2,  x=y_valid, c='green', s=7)
plt.plot([0,3e6],[0, 3e6], 'r-')
plt.legend(['"Ideal model" diagonal','Predicted train data','Predicted validation data'])
plt.text(s=" MAE_train = {0:.2f}".format(mae(y_pred_train_2, y_train)), x=1e5, y=3e6)
plt.text(s=" MAE_test = {0:.2f}".format(mae(y_pred_valid_2, y_valid)),  x=1e5, y=2.9e6)
plt.text(s="""$ \\lambda = {0:.4f} $""".format(model_grid.best_estimator_.steps[1][1].alpha), x=1e5, y=2.8e6)
plt.title("True VS Predicted values (LASSO with additional features)")
plt.grid()


والآن، حصلنا على $MAE_{train} = 211470$ و$MAE_{valid} = 208750$. 
نرى أن هذا الخطأ في مجموعة التحقق من الصحة قد انخفض بنسبة **3.8%**



الآن، دعونا نجرب RandomForest مع المعلمات الافتراضية على نفس الميزات (ولكن بدون ميزات متعددة الحدود):


In [ ]:
%%time
rf = RandomForestRegressor(n_estimators=300, random_state=42).fit(X_ohe_train_rf, to_log(y_train_2))

In [ ]:
y_pred_train_rf = from_log(rf.predict(X_ohe_train_rf))
y_pred_valid_rf = from_log(rf.predict(X_ohe_valid_rf))

In [ ]:
plt.figure(figsize=(12,12))
plt.xlim((0, y.quantile(0.99)))
plt.ylim((0, y.quantile(0.99)))
plt.scatter(y=y_pred_train_2,  x=y_train, c='blue', s=7)
plt.scatter(y=y_pred_valid_2,  x=y_valid, c='green', s=7)
plt.plot([0,3e6],[0, 3e6], 'r-')
plt.legend(['"Ideal model" diagonal','Predicted train data','Predicted validation data'])
plt.text(s=" MAE_train = {0:.2f}".format(mae(y_pred_train_rf, y_train)), x=1e5, y=3e6)
plt.text(s=" MAE_test = {0:.2f}".format(mae(y_pred_valid_rf, y_valid)),  x=1e5, y=2.9e6)
plt.text(s="max_depth = {0} ".format(rf.max_depth), x=1e5, y=2.8e6)
plt.title("True VS Predicted values (RandomForest)")
plt.grid()


لقد حصلنا على **نموذج overfitted**، لأن الخطأ في مجموعة بيانات القطار أقل بأكثر من الضعف (!) من الخطأ في مجموعة التحقق من الصحة.لذا، دعونا نحاول ضبط المعلمات الفائقة لـ RandomForest باستخدام التحقق المتبادل للحصول على جودة أفضل.



### الجزء العاشر. رسم منحنيات التدريب والتحقق من الصحة



سنوفر التحقق المتبادل لـ RandomForest باستخدام وحدة **cross_val_score** مع إستراتيجية **TimeSeriesSplit** وn_splits=5. سنستخدم أيضًا **المسجل** الخاص بنا - غلاف دالة MAE لاستخدامه في وظيفة cross_val_score. معلمة الضبط لدينا هي **max_deep**.


In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
from sklearn.metrics import make_scorer

# Create scorer with our MAE-function
scorer = make_scorer(mae, greater_is_better=False)

In [ ]:
# MAE-function for logarithmic inputs
def mae_score(y_true, y_pred):
    return mae(from_log(y_true), from_log(y_pred))

In [ ]:
scorer = make_scorer(mae_score)

In [ ]:
# List of params values
max_depth_list = [10, 12, 13, 15, 17, 20, 25]


أثناء عملية التحقق المتبادل لكل قيمة معلمة، يتم تقسيم مجموعة بيانات القطار الخاصة بنا إلى 5 طيات: أربعة لملاءمة النموذج وواحدة للحصول على خطأ التحقق المتبادل. لذا، سنحصل على قائمة بالأخطاء الأربعة الأولى، ونحسب متوسطها وسيكون **'Cross validation MAE on Train'** لقيمة max_عمق الحالية. بعد ذلك، سنقوم بالتنبؤ بمجموعة التحقق من الصحة بنفس قيمة max_degree وحساب **'MAE عند مجموعة التحقق من الصحة'**. التنبؤ *على مجموعة بيانات القطار الكاملة* (بدون التقسيم على الطيات) سوف يعطينا **'MAE في مجموعة القطار'**.


In [ ]:
%%time
cv_errors_list = []
train_errors_list = []
valid_errors_list = []

for max_depth in max_depth_list:
    rf = RandomForestRegressor(n_estimators=300, max_depth=max_depth,random_state=42)

    
    cv_errors = cross_val_score(estimator=rf, 
                                  X=X_ohe_train_rf, 
                                  y=to_log(y_train_2), 
                                  scoring=scorer,
                                  cv=TimeSeriesSplit(n_splits=5))  
    cv_errors_list.append(cv_errors.mean())
    
    rf.fit(X=X_ohe_train_rf, y=to_log(y_train_2))
    
    valid_error = mae_score(to_log(y_valid_2), rf.predict(X_ohe_valid_rf))    
    valid_errors_list.append(valid_error)
    
    train_error = mae_score(to_log(y_train_2), rf.predict(X_ohe_train_rf))
    train_errors_list.append(train_error)
    
    print(max_depth)
    


لنرسم الآن تبعيات الأخطاء أعلاه من معلمة **max_deep**:


In [ ]:
plt.figure(figsize=(10, 7))

plt.plot(max_depth_list,cv_errors_list)
plt.plot(max_depth_list,train_errors_list)
plt.plot(max_depth_list,valid_errors_list)
plt.vlines(x=max_depth_list[np.array(cv_errors_list).argmin()], 
           ymin=0, ymax=2e5, 
           linestyles='dashed', colors='r')

plt.legend(['Cross validation MAE on train', 
            'MAE on train set', 
            'MAE on validation set', 
            'Best Max_depth value on CV'])
plt.title("MAE on train and validation sets.")
plt.xlabel('Max_depth value')
plt.ylabel('MAE value')
plt.grid()


من الواضح أنه في البداية تتناقص جميع الأخطاء. ولكن بعد قيمة **max_Deep** المحددة **'التحقق المتقاطع MAE في القطار'** و **'MAE في مجموعة التحقق من الصحة'** تتزايد. يتم توفير الحد الأدنى لقيمة هذه الأخطاء من خلال **max_عمق**، مع وضع علامة بخط أحمر متقطع وأقصى عمق = 15.


In [ ]:
best_max_depth


لاحظ أنه في أول تركيب RandomForest حصلنا على **max_عمق = 25**:


In [ ]:
rf.max_depth


### الجزء 11. التنبؤ بالعينات الاختبارية أو المحتجزة



في النهاية، دعونا نحاول ملاءمة RandomForestRegressor مع أفضل قيمة **max_Deep**، التي تم العثور عليها نتيجة للتحقق المتبادل. سوف نستخدم نفس مجموعات البيانات.


In [ ]:
rf_best = RandomForestRegressor(n_estimators=300, max_depth=best_max_depth, random_state=42)

In [ ]:
rf_best.fit(X_ohe_train_rf, to_log(y_train_2))

كما هو الحال سابقًا، فلنقم بالتنبؤ بمجموعة التحقق من الصحة ورسم القيم الحقيقية والمتوقعة في مجموعات بيانات التدريب والتحقق من الصحة:


In [ ]:
y_pred_train_rf_best = from_log(rf_best.predict(X_ohe_train_rf))
y_pred_valid_rf_best = from_log(rf_best.predict(X_ohe_valid_rf))

In [ ]:
plt.figure(figsize=(12,12))
plt.xlim((0, y.quantile(0.99)))
plt.ylim((0, y.quantile(0.99)))
plt.scatter(y=y_pred_train_2,  x=y_train, c='blue', s=7)
plt.scatter(y=y_pred_valid_2,  x=y_valid, c='green', s=7)
plt.plot([0,3e6],[0, 3e6], 'r-')
plt.legend(['"Ideal model" diagonal','Predicted train data','Predicted validation data'])
plt.text(s=" MAE_train = {0:.2f}".format(mae(y_pred_train_rf_best, y_train)), x=1e5, y=3e6)
plt.text(s=" MAE_test = {0:.2f}".format(mae(y_pred_valid_rf_best, y_valid)),  x=1e5, y=2.9e6)
plt.text(s="max_depth = {0} ".format(rf_best.max_depth), x=1e5, y=2.8e6)
plt.title("True VS Predicted values (RandomForest afret max_depth tuning)")
plt.grid()


لذلك، يتم زيادة MAE في القطار بشكل ملحوظ، ولكن يتم تقليل MAE عند التحقق بشكل طفيف. وهذا يعني أن نموذجنا الآن لم يعد مُجهزًا بشكل زائد كما كان من قبل ولديه قدرة تعميم أكبر.



### الجزء 12. الاستنتاجات



قد يكون الحل مفيدًا **للوكالات العقارية**، التي تجمع مثل هذه البيانات وتحاول التنبؤ بالأسعار الأكثر ملائمة للعقارات. إنه أمر مهم، لأنه يسمح لهم ببيع/شراء العقارات في أسرع وقت ممكن دون خسائر مالية ومع الحفاظ على ولاء العملاء.
الحالات المحتملة لتحسين النموذج هي إنشاء ميزات وتجارب أكثر فائدة مع أنواع أخرى من النماذج، على سبيل المثال، استخدام تعزيز التدرج (xgboost، LightGBM، CatBoost)